#### Key Variables

See link for variable descriptions: https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/NHIS/2023/adult-summary.pdf


| Variable         | Description                                        |
|-----------------|----------------------------------------------------|
| SRVY_YR         | Year of the National Health Interview Survey       |
| EVERCOVD_A      | Ever had COVID-19                                  |
| SHTCVD191_A     | COVID-19 vaccination                                |
| EMPDYSMSS3_A    | Days missed work, past 12 months (top-coded)      |
| HICOV_A         | Have health insurance                             |
| EMDINDSTN1_A    | Detailed 2-digit recode for sample adult's industry |
| SEX_A           | Sex of Sample Adult                               |
| AGEP_A          | Age of SA (top coded)                            |
| EDUCP_A         | Educational level of sample adult                |
| REGION          | Household region                                 |

In [1]:
# Setup AWS and SageMaker session
import boto3
import sagemaker
import pandas as pd
import io
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

/opt/conda/lib/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
# Initialize SageMaker session and environment variables
sess = sagemaker.Session()
bucket = "jc-bucket-ads508-2025-finalproject"
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
sm = boto3.client("sagemaker", region_name=region)

In [3]:
# Define S3 paths
s3_public_path = "s3://jc-bucket-ads508-2025-finalproject/ld_data/"
s3_private_path = f"s3://{bucket}/ld_data"
s3_processed_path = f"s3://{bucket}/ld_data_processed"

In [4]:
# Store paths for later use
%store s3_public_path
%store s3_private_path
%store s3_processed_path

Stored 's3_public_path' (str)
Stored 's3_private_path' (str)
Stored 's3_processed_path' (str)


In [5]:
# Copy CSV files from public to private bucket
!aws s3 cp --recursive {s3_public_path} {s3_private_path}/ --include "*.csv" > /dev/null 2>&1

In [6]:
# Define the improved standardize_column_names function
def standardize_column_names(df, year):
    # Create a copy to avoid modifying the original
    df_standardized = df.copy()
    
    # COVID-19 diagnosis variable standardization
    if year in [2020, 2021, 2022]:
        if 'CVDDIAG_A' in df.columns:
            df_standardized['EVERCOVD_A'] = df['CVDDIAG_A']
    elif year not in [2023]:  # In 2019, COVID didn't exist
        df_standardized['EVERCOVD_A'] = None  # Add column with nulls for 2019
    
    # COVID-19 vaccination standardization
    if year == 2021:
        if 'SHTCVD19_A' in df.columns:
            df_standardized['SHTCVD191_A'] = df['SHTCVD19_A']
    elif year not in [2022, 2023]:  # In 2019-2020, no COVID vaccination
        df_standardized['SHTCVD191_A'] = None  # Add column with nulls
    
    # Work days missed standardization
    if year in [2019, 2020]:
        if 'EMPDYSMSS2_A' in df.columns:
            df_standardized['EMPDYSMSS3_A'] = df['EMPDYSMSS2_A']
    
    # Industry code standardization
    if year == 2020:
        if 'EMDINDSTR1_A' in df.columns:
            df_standardized['EMDINDSTN1_A'] = df['EMDINDSTR1_A']
    elif year not in [2021, 2023]:  # Handle years where this var doesn't exist
        df_standardized['EMDINDSTN1_A'] = None
    
    # Education standardization
    if year in [2019, 2020]:
        if 'EDUC_A' in df.columns:
            df_standardized['EDUCP_A'] = df['EDUC_A']
    
    # Region - check if it exists for the later years
    if 'REGION' not in df.columns and year in [2021, 2022, 2023]:
        df_standardized['REGION'] = None
    
    # Select only the columns we need, handling missing columns
    columns_to_select = ['SRVY_YR', 'EVERCOVD_A', 'SHTCVD191_A', 'EMPDYSMSS3_A', 
                         'HICOV_A', 'EMDINDSTN1_A', 'SEX_A', 'AGEP_A', 'EDUCP_A', 'REGION']
    
    # Create a list of columns that exist in the dataframe
    existing_columns = [col for col in columns_to_select if col in df_standardized.columns]
    
    # Return only columns that exist
    return df_standardized[existing_columns]

In [7]:
# Process all CSV files and save to processed directory
s3 = boto3.resource('s3')
s3_client = boto3.client('s3')

In [8]:
# Check if we can determine the year from the filename
def extract_year_from_filename(filename):
    for y in ['2019', '2020', '2021', '2022', '2023']:
        if y in filename:
            return int(y)
    return None

In [9]:
# Create a function to process and upload each file
def process_csv_file(bucket_name, file_key, processed_prefix):
    try:
        # Get the file content
        obj = s3.Object(bucket_name, file_key).get()
        df = pd.read_csv(io.BytesIO(obj['Body'].read()))
        
        # Try to extract year from filename or from SRVY_YR column
        year = None
        if 'SRVY_YR' in df.columns:
            year = int(df['SRVY_YR'].iloc[0])
        else:
            # Try to extract from filename
            filename = file_key.split('/')[-1]
            year = extract_year_from_filename(filename)
            if year:
                df['SRVY_YR'] = year
        
        if year is None:
            print(f"Warning: Could not determine year for {file_key}, skipping")
            return
        
        print(f"Processing {file_key} (Year: {year})")
        
        # Standardize column names
        processed_df = standardize_column_names(df, year)
        
        # Ensure all required columns exist (fill with NULL if missing)
        required_columns = ['SRVY_YR', 'EVERCOVD_A', 'SHTCVD191_A', 'EMPDYSMSS3_A', 
                           'HICOV_A', 'EMDINDSTN1_A', 'SEX_A', 'AGEP_A', 'EDUCP_A', 'REGION']
        
        for col in required_columns:
            if col not in processed_df.columns:
                processed_df[col] = None
        
        # Reorder columns to match the Athena table schema
        processed_df = processed_df[required_columns]
        
        # Save processed file
        filename = file_key.split('/')[-1]
        processed_key = f"{processed_prefix}/{filename}"
        
        csv_buffer = io.StringIO()
        processed_df.to_csv(csv_buffer, index=False)
        s3_client.put_object(Bucket=bucket_name, 
                           Key=processed_key, 
                           Body=csv_buffer.getvalue())
        
        print(f"Saved processed file to {processed_key}")
        return True
    except Exception as e:
        print(f"Error processing {file_key}: {e}")
        return False

# List files in the bucket and process each one
print("Processing CSV files...")
processed_prefix = "ld_data_processed"
response = s3_client.list_objects_v2(Bucket=bucket, Prefix="ld_data")

if 'Contents' in response:
    success_count = 0
    for obj in response['Contents']:
        if obj['Key'].endswith('.csv'):
            if process_csv_file(bucket, obj['Key'], processed_prefix):
                success_count += 1
    
    print(f"Successfully processed {success_count} files")
else:
    print("No files found in the source location")

Processing CSV files...
Processing ld_data/adult19.csv (Year: 2019)
Saved processed file to ld_data_processed/adult19.csv
Processing ld_data/adult20.csv (Year: 2020)
Saved processed file to ld_data_processed/adult20.csv
Processing ld_data/adult21.csv (Year: 2021)
Saved processed file to ld_data_processed/adult21.csv
Processing ld_data/adult22.csv (Year: 2022)
Saved processed file to ld_data_processed/adult22.csv
Processing ld_data/adult23.csv (Year: 2023)
Saved processed file to ld_data_processed/adult23.csv
Successfully processed 5 files


In [10]:
# --- Create Athena Database ---
database_name = "demo2"
s3_staging_dir = f"s3://{bucket}/athena/staging"

# Connect to Athena
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

# Create database
create_db_statement = f"CREATE DATABASE IF NOT EXISTS {database_name}"
print(create_db_statement)
pd.read_sql(create_db_statement, conn)

# Verify database creation
show_db_statement = "SHOW DATABASES"
df_show = pd.read_sql(show_db_statement, conn)
print(df_show.head(5))

# Mark database creation as successful if database exists
ingest_create_athena_db_passed = database_name in df_show['database_name'].values
print(f"Database creation successful: {ingest_create_athena_db_passed}")
%store ingest_create_athena_db_passed

CREATE DATABASE IF NOT EXISTS demo2
  database_name
0       default
1         demo1
2         demo2
3         demo3
Database creation successful: True
Stored 'ingest_create_athena_db_passed' (bool)


In [11]:
# --- Create Athena Table using processed data ---
if not ingest_create_athena_db_passed:
    print("[ERROR] You need to create the Athena database first")
else:
    # Define table
    table_name = "ld_data"
    
    # Drop table if it exists to recreate it
    drop_table_statement = f"DROP TABLE IF EXISTS {database_name}.{table_name}"
    pd.read_sql(drop_table_statement, conn)
    
    # Statement to execute with actual path to processed data
    create_table_statement = f"""CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.{table_name}(
        SRVY_YR int,
        EVERCOVD_A int,
        SHTCVD191_A int,
        EMPDYSMSS3_A int,
        HICOV_A int,
        EMDINDSTN1_A string,
        SEX_A int,
        AGEP_A int,
        EDUCP_A string,
        REGION int
    ) ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' LINES TERMINATED BY '\\n' 
    LOCATION '{s3_processed_path}'
    TBLPROPERTIES ('skip.header.line.count'='1')"""
    
    # Print statement with redacted sensitive information
    print_statement = create_table_statement.replace(s3_processed_path, "s3://[REDACTED_BUCKET]/[REDACTED_PATH]")
    print(print_statement)
    
    # Execute statement
    pd.read_sql(create_table_statement, conn)
    
    # Verify table creation
    show_tables_statement = f"SHOW TABLES in {database_name}"
    df_show = pd.read_sql(show_tables_statement, conn)
    print(df_show.head(5))
    
    # Mark table creation as successful if table exists
    ingest_create_athena_table_passed = table_name in df_show['tab_name'].values
    print(f"Table creation successful: {ingest_create_athena_table_passed}")
    %store ingest_create_athena_table_passed

CREATE EXTERNAL TABLE IF NOT EXISTS demo2.ld_data(
        SRVY_YR int,
        EVERCOVD_A int,
        SHTCVD191_A int,
        EMPDYSMSS3_A int,
        HICOV_A int,
        EMDINDSTN1_A string,
        SEX_A int,
        AGEP_A int,
        EDUCP_A string,
        REGION int
    ) ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' LINES TERMINATED BY '\n' 
    LOCATION 's3://[REDACTED_BUCKET]/[REDACTED_PATH]'
    TBLPROPERTIES ('skip.header.line.count'='1')
  tab_name
0  ld_data
Table creation successful: True
Stored 'ingest_create_athena_table_passed' (bool)


In [12]:
# Query to count total records in the table
count_query = f"""
SELECT COUNT(*) as total_records 
FROM {database_name}.{table_name}
"""
print("Query to count total records:")
print(count_query)

# Execute the query
df_count = pd.read_sql(count_query, conn)

# Display the result
print(f"Total records in {table_name}: {df_count['total_records'].iloc[0]}")

# Query to select the first 10 rows from the table
sample_query = f"""
SELECT *
FROM {database_name}.{table_name}
LIMIT 10
"""

print("Query to retrieve sample records:")
print(sample_query)

# Execute the query
df_sample = pd.read_sql(sample_query, conn)

# Display the result
print(f"\nSample records from {table_name}:")
display(df_sample)  # If in Jupyter notebook, or use print(df_sample) otherwise

Query to count total records:

SELECT COUNT(*) as total_records 
FROM demo2.ld_data

Total records in ld_data: 150220
Query to retrieve sample records:

SELECT *
FROM demo2.ld_data
LIMIT 10


Sample records from ld_data:


,srvy_yr,evercovd_a,shtcvd191_a,empdysmss3_a,hicov_a,emdindstn1_a,sex_a,agep_a,educp_a,region
0,2019,None,None,NaN,1,,1,97,7,3
1,2019,None,None,0.0,1,,2,28,6,3
2,2019,None,None,NaN,1,,1,72,5,3
3,2019,None,None,0.0,1,,1,60,7,3
4,2019,None,None,0.0,1,,1,60,5,3
5,2019,None,None,NaN,1,,1,78,5,3
6,2019,None,None,0.0,1,,1,32,8,3
7,2019,None,None,NaN,2,,1,61,4,3
8,2019,None,None,3.0,1,,1,28,5,3
9,2019,None,None,NaN,1,,1,85,1,3


In [13]:
!aws s3 ls s3://jc-bucket-ads508-2025-finalproject/ --recursive

2025-03-22 06:27:48    8235550 ab_data/analytic_data2019.csv
2025-03-22 06:27:49   12162745 ab_data/analytic_data2020_0.csv
2025-03-22 06:27:51   12478210 ab_data/analytic_data2021.csv
2025-03-22 06:27:52   12760127 ab_data/analytic_data2022.csv
2025-03-22 06:27:53   12525733 ab_data/analytic_data2023_0.csv
2025-03-22 22:35:28    3139006 ab_data/analytic_data_2019_2023_cleaned.csv
2025-03-24 01:10:47          0 athena/staging/065a38c3-68e8-40d0-b39e-98cbca045a41.txt
2025-03-24 01:10:44          0 athena/staging/1cc9df8c-21a2-4a4b-848a-4dadd4a55c17.txt
2025-03-24 01:10:45         25 athena/staging/21114e26-47a1-4cd2-b652-c73b5f042df9.txt
2025-03-24 01:10:45        312 athena/staging/21114e26-47a1-4cd2-b652-c73b5f042df9.txt.metadata
2025-03-24 01:10:50         25 athena/staging/ac730c17-c900-45ea-b4f5-3131b6a29402.csv
2025-03-24 01:10:50         83 athena/staging/ac730c17-c900-45ea-b4f5-3131b6a29402.csv.metadata
2025-03-24 01:10:51        468 athena/staging/af42594b-4b7f-47de-8a39-59280e